In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model 
import pickle 
import pandas as pd
import numpy as np 


In [3]:
# Load the trained model,scaler pickle,onehot 
model = load_model('model.h5')

## load the encoder and scaler 
with open('onehot_encoder_geo.pkl','rb') as file: 
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file: 
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [22]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [23]:
input_df = pd.DataFrame.from_dict([input_data],orient='columns')

In [24]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [25]:
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]])
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [26]:
input_df['Gender'] = label_encoder_gender.transform([input_df['Gender']])

d:\Machine_Learning\Python\Section_53\by_me\venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


In [27]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [28]:
#Concatination of one hot encoded 
input_df = pd.concat([input_df.drop(columns='Geography'),geo_encoded_df],axis=1)

In [29]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [30]:
# Scaling the input data 
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.51254662,  0.91647499,  0.11671159, -0.71668386, -0.30785754,
         0.83654433,  0.66167554,  0.95692675, -0.86916204,  1.03046381,
        -0.59120865, -0.58658846]])

In [31]:
# Predicting churn 
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


array([[0.02328688]], dtype=float32)

In [32]:
prediction_proba = prediction[0][0]

In [34]:
print(prediction_proba)

0.02328688


In [35]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
